## Amdahl plots — one figure per `.dat` in this folder

Each `.dat` file has rows: `n c Ax acc0 trans threads backend mean stdev se`.
For each file we produce a side-by-side figure with runtime-vs-threads (log-log)
and speedup-vs-threads (with ideal `y=x` reference).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

COLUMNS = ['n', 'c', 'Ax', 'acc0', 'trans', 'threads', 'backend', 'mean', 'stdev', 'se']

def load(path):
    df = pd.read_csv(path, sep=r'\s+', header=None, names=COLUMNS)
    return df.sort_values(['c', 'threads']).reset_index(drop=True)

def plot_amdahl(df, title):
    fig, (ax_rt, ax_sp) = plt.subplots(1, 2, figsize=(12, 5))
    cs = sorted(df['c'].unique())
    for c in cs:
        sub = df[df['c'] == c].sort_values('threads')
        # runtime vs threads
        ax_rt.errorbar(sub['threads'], sub['mean'], yerr=sub['stdev'],
                       marker='o', capsize=3, label=f'c={c}')
        # speedup vs threads
        baseline = sub[sub['threads'] == sub['threads'].min()]['mean'].iloc[0]
        speedup = baseline / sub['mean']
        ax_sp.plot(sub['threads'], speedup, marker='o', label=f'c={c}')
    # ideal speedup reference
    tmin, tmax = df['threads'].min(), df['threads'].max()
    ax_sp.plot([tmin, tmax], [tmin, tmax], 'k--', alpha=0.4, label='ideal (y=x)')
    # axes
    ax_rt.set_xscale('log'); ax_rt.set_yscale('log')
    ax_rt.set_xlabel('threads'); ax_rt.set_ylabel('runtime (s)')
    ax_rt.set_title('Runtime')
    ax_rt.grid(True, which='both', alpha=0.3)
    ax_rt.legend()
    ax_sp.set_xscale('log'); ax_sp.set_yscale('log')
    ax_sp.set_xlabel('threads'); ax_sp.set_ylabel('speedup vs 1-thread baseline')
    ax_sp.set_title('Speedup')
    ax_sp.grid(True, which='both', alpha=0.3)
    ax_sp.legend()
    fig.suptitle(title)
    fig.tight_layout()
    return fig

def plot_speedup_linear(df):
    """Report-facing speedup figure: x=threads (log, with explicit tick labels),
    y=speedup (linear). No title (caption goes in LaTeX). No ideal-line reference.
    Legend uses J= notation matching the report."""
    fig, ax = plt.subplots(figsize=(7, 5))
    cs = sorted(df['c'].unique())
    for c in cs:
        sub = df[df['c'] == c].sort_values('threads')
        baseline = sub[sub['threads'] == sub['threads'].min()]['mean'].iloc[0]
        speedup = baseline / sub['mean']
        ax.plot(sub['threads'], speedup, marker='o', label=f'J={c}')
    ax.set_xscale('log')
    # Explicit tick labels at the actual thread counts, with plain (non-exponent) formatting
    threads = sorted(df['threads'].unique())
    ax.set_xticks(threads)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.minorticks_off()
    ax.set_xlabel('Worker threads')
    ax.set_ylabel('Speedup relative to one thread')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    return fig

In [ ]:
here = Path('.')
dats = sorted(here.glob('*.dat'))
if not dats:
    print('No .dat files found in', here.resolve())
for p in dats:
    print(f'=== {p.name} ===')
    df = load(p)
    display(df)
    plot_amdahl(df, p.stem)
    plt.show()
    plot_speedup_linear(df)
    plt.show()